In [ ]:
import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
from huggingface_hub import hf_hub_download
torch.set_float32_matmul_precision('high')

# --- download model ---

path = hf_hub_download(repo_id="Ksgk-fy/sorl", filename="ts-k4-v128.pt", repo_type="model")
path = hf_hub_download(
    repo_id="Ksgk-fy/sorl",
    filename="ts-k4-v128.pt",
    repo_type="model",
    local_dir="./ckpt",  # saves to ./ckpt/ts-k4-v128.pt
)

gat_config = GATConfig.gpt_size("small", [BOS_TOKEN_ID+1, 128])
model = GAT(gat_config)

# ---- load ckpt ----
def local_load_ckpt(model, ckpt_path):
    """load compiled ckpt locally"""
    model_ckpt = torch.load(ckpt_path, map_location="cpu")['model']
    clean_state_dict = {k.replace("_orig_mod.", ""): v for k, v in model_ckpt.items()}
    model.load_state_dict(clean_state_dict)
    return model

ckpt_path = "ckpt/ts-k4-v128.pt"
model = local_load_ckpt(model, ckpt_path)
K = 4

In [4]:
from data.tinystory_local import TinyStoriesDataLoader, TinyStoriesDataLoader_v2, collect_rollout_statistics, AbstractionStatistics
from data.tinystory_local import visualize_dynamics
import tiktoken 

# TinyStories Dataset + GPT2 tokenizer
# -------------------------------------
num_stories = 100
max_len = 128
doc_len = max_len  + (max_len - 1) // K # <-- doc len contains abstract tokens
pad_shift = 2
loader = TinyStoriesDataLoader(num_stories=num_stories, max_len=max_len, chunk_size=K, device='cpu', split="validation", pad_shift=pad_shift)

batch_size = 8
memory_span = 1792
attn_blocksize = 1792
max_iterations = 2

# ---- stat collection ---
abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# --- tokenizer --- 
enc = tiktoken.get_encoding("gpt2")
eot = enc._special_tokens['<|endoftext|>']

Loading 100 stories from TinyStories validation...
Loaded 100 stories, 12554 tokens total, 0.05 MB
Collected 2707 unique 4-chunks


In [6]:
# (I). Generate with SoRL trained on TinyStories Dataset
# -------------------------------------------------
from sorl.neo_utils import generate
from data.tinystory_local import visualize_interleaved_alignment    

min_temperature = 0.0
tokens, doc_ids = loader.get_batch(1)

# idx = tokens[:, :15].clone()
idx = torch.tensor(enc.encode("Chrismas is coming soon, ")).unsqueeze(0)

img_frames = []
for i in range(120): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, temperature=torch.tensor(min_temperature))
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    img = visualize_interleaved_alignment(idx, model, enc, K=K, max_chunks=8)
    img_frames.append(img)

# ---- save to gif ----
if len(img_frames) > 0:
    img_frames[0].save(
        'tiny-tiny-stories-generation.gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds sper frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

Saved GIF with 120 frames


In [8]:
from huggingface_hub import HfApi

# ---- upload ckpt to huggingface ----

api = HfApi()
api.create_repo(repo_id="Ksgk-fy/sorl", repo_type="model", private=False)

api.upload_file(
    path_or_fileobj="ckpt/ts-k4-v128.pt",
    path_in_repo="ts-k4-v128.pt",
    repo_id="Ksgk-fy/sorl",
    repo_type="model"
)

# api.upload_file(
#     path_or_fileobj="ckpt/gpt2-small-ts.pt",
#     path_in_repo="gpt2-small-ts.pt",
#     repo_id="Ksgk-fy/sorl",
#     repo_type="model"
# )

HfHubHTTPError: Client error '409 Conflict' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-698a9ac8-5ff8aa49628993be0e51f4cc;2a555444-f48f-427a-96ec-62d80291be42)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/409

You already created this model repo: Ksgk-fy/sorl

In [10]:
from sorl.forget import compute_abs_stats_v4
from tqdm import tqdm
from collections import defaultdict

n = 5
temperature = torch.tensor([0.0] + [5.0] * (n - 1))
batch_size = 2

# Collect all stats
all_stats = defaultdict(list)

with tqdm(total=len(loader.stories), desc="Processing batches") as pbar:
    for i in range(0, len(loader.stories), batch_size):
        batch_indices = torch.arange(i, min(i + batch_size, len(loader.stories)))
        tokens, doc_ids = loader.get_specific(batch_indices)

        stat_dict = compute_abs_stats_v4(
            tokens, model, base_model, n=n, K=K, max_iterations=max_iterations,
            memory_span=memory_span, attn_blocksize=attn_blocksize,
            temperature=temperature, truncate_seq_len=False, pad_token=loader.pad_token
        )

        for key, val in stat_dict.items():
            all_stats[key].append(val.item() if hasattr(val, 'item') else val)

        pbar.update(len(batch_indices))
        break

# Print averages
for key, vals in all_stats.items():
    print(f"{key}: {sum(vals) / len(vals):.6f}")

Processing batches:   2%|▏         | 2/100 [00:02<01:41,  1.04s/it]

id_base_traj_loss: 1.316271
id_greedy_traj_loss: 1.227240
id_greedy_info_gain: 0.089031
id_search_info_gain: 0.091862
id_greedy_adv: 0.306241
alien_base_traj_loss: 19.114380
alien_greedy_traj_loss: 22.920513
alien_greedy_info_gain: -3.806132
alien_search_info_gain: -3.806132
alien_greedy_adv: 0.021730


In [ ]:
from sorl.forget import collect_forget_data, plot_normalized_correlation_lines, train_forget_vec
from tqdm import tqdm as tqdm

# ---- Memory Interference Experiment ---

abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

abs_stats_post = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# ---- Compute 'Forget Matrix' --- 
forget_mat = torch.zeros(num_stories, num_stories, device=model.device)
num_steps = 40
for train_idx in tqdm(range(num_stories)): 
    forget_vec = train_forget_vec(train_idx, loader, model, abs_stats, abs_stats_post, optimizer, num_steps,
                                max_iterations, memory_span, attn_blocksize, temperature, K, r_min, reward_mode, loss_fn, alpha_abs, alpha_soft_zipf, alpha_topo,
                                ckpt_path="sorl_tinystories.pt")
    forget_mat[train_idx] = forget_vec 

# torch.save(forget_mat, "forget_mat.pt") # save it just in case

# # ---- Visualize 'Forget Matrix' --- 
# correlation_data, ham_corrs = collect_forget_data(forget_mat, abs_stats)

# plot_normalized_correlation_lines(
#     correlation_data, 
#     ham_corrs,
#     xlabel="Abstraction Edit Distance", 
#     ylabel="Forgetting (Δ Perplexity)",
#     title="Distant Abstractions → Less Forgetting (TinyStories)"
# )

In [ ]:
from data.tinystory_local import *
from sklearn.decomposition import PCA

cs_sim = abs_stats.compute_cross_doc_logit_sim()
# cs_sim = abs_stats.compute_cross_doc_hamming()
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(cs_sim.cpu().numpy())

# Now use in 3D visualization
train_idx = 0
perplexity = forget_mat[train_idx]
img = visualize_forget_terrain(coords_2d, perplexity, trained_idx=train_idx, step=step)
# img = visualize_perplexity_terrain(coords_2d, perplexity, trained_idx=train_idx, step=step)

In [4]:
# Hypothesis #2. 
# ------------------------------------------------------------
# forget(i | j) is proportional to 1 / d(a_i, a_j)
# dis-similar concept is less likely to be overwritten by each other
# similar concept is more likely to be overwritten by each other
# the emerged abstraction system from SoRL can describe such similarity via d(a_i, a_j)
# ------------------------------------------------------------

# Experiment 
# (a). Train till emergence
# (b). Train with specific order. 
# (c). Record 'forgetting matrix'

import numpy as np
np.array(record['topo_loss'][:5]).mean(), np.array(record['topo_loss'][-5:]).mean()

(-0.7774280548095703, -0.7865824460983276)

In [4]:
if len(img_frames) > 0:
    img_frames[0].save(
        'tinystories_dynamics (select-best per abs SoRL + 1.0 topo reg + 1.0 bigram zipf reg + utility reward scaling.gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds per frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

Saved GIF with 200 frames


In [ ]:
# Generate with SoRL trained on TinyStories Dataset
# -------------------------------------------------
from sorl.neo_utils import generate
from data.tinystory_local import visualize_interleaved_alignment    

min_temperature = 0.0
tokens, doc_ids = loader.get_batch(1)

idx = tokens[:, :15].clone()

img_frames = []
for i in range(30): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, temperature=torch.tensor(min_temperature))
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    img = visualize_interleaved_alignment(idx, model, enc, K=K, max_chunks=8)
    img_frames.append(img)

# ---- save to gif ----
if len(img_frames) > 0:
    img_frames[0].save(
        'tiny-tiny-stories-generation.gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds per frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

In [ ]:
# Request #1. 
# -> Full scale experiment on TinyStories & FineWeb